# Test api anthropic

In [2]:
import anthropic
import pdfplumber
import regex as re
import json
import ipywidgets as widgets
from IPython.display import display

In [3]:
pdf = pdfplumber.open("data/luth_full.pdf")

first_index_page = None
last_index_page = None

for n, page in enumerate(pdf.pages):

    if first_index_page and last_index_page:
        break

    text = page.extract_text()

    if "Contents" in text and not first_index_page:
        first_index_page = n
    

    if "Index" in text and not last_index_page:
        last_index_page = n


print(f"First index page: {first_index_page}")
print(f"Last index page: {last_index_page}")

index_text = ""
for n in range(first_index_page, last_index_page + 1):
    page = pdf.pages[n]
    text = page.extract_text(x_tolerance=2)
    index_text += text + "\n"
    print(text)


First index page: 9
Last index page: 13
Contents
1 Surface and Interface Physics: Its Definition and Importance . . . 1
Panel I: Ultrahigh Vacuum (UHV) Technology . . . . . . . . . . 6
Panel II: Basics of Particle Optics and Spectroscopy . . . . . . . . 17
Problems . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 28
2 Preparation of Well-Defined Surfaces, Interfaces and Thin Films . 29
2.1 Why Is Ultrahigh Vacuum Used? . . . . . . . . . . . . . . . . . 29
2.2 Cleavage in UHV . . . . . . . . . . . . . . . . . . . . . . . . . 31
2.3 Ion Bombardment and Annealing . . . . . . . . . . . . . . . . . 34
2.4 Evaporation and Molecular Beam Epitaxy (MBE) . . . . . . . . 35
2.5 Epitaxy by Means of Chemical Reactions . . . . . . . . . . . . . 43
Panel III: Auger Electron Spectroscopy (AES) . . . . . . . . . . . 49
Panel IV: Secondary Ion Mass Spectroscopy (SIMS) . . . . . . . . 55
Problems . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 63
3 Morphology and Struct

In [86]:
panels_regex = r"(?:Panel\s+[IVX]+)\s*:"
section_regex = r"(?:\d+(?:\.\d+)*)"
problems_regex = r"(?:Problems\s*)"
name_regex = r"(.+?)\s*\."
dots_regex = r"(?:\s*\.)*"
page_regex = r"(\d+)$"

chapter_regex = re.compile(
    fr"^({panels_regex}|{section_regex})\s+{name_regex}\s+{dots_regex}\s+{page_regex}$",
    re.MULTILINE
)

matches = chapter_regex.findall(index_text)

for match in matches:
    print(match)

('1', 'Surface and Interface Physics: Its Definition and Importance', '1')
('Panel I:', 'Ultrahigh Vacuum (UHV) Technology', '6')
('Panel II:', 'Basics of Particle Optics and Spectroscopy', '17')
('2.1', 'Why Is Ultrahigh Vacuum Used?', '29')
('2.2', 'Cleavage in UHV', '31')
('2.3', 'Ion Bombardment and Annealing', '34')
('2.4', 'Evaporation and Molecular Beam Epitaxy (MBE)', '35')
('2.5', 'Epitaxy by Means of Chemical Reactions', '43')
('Panel III:', 'Auger Electron Spectroscopy (AES)', '49')
('Panel IV:', 'Secondary Ion Mass Spectroscopy (SIMS)', '55')
('3.1', 'Surface Stress, Surface Energy, and Macroscopic Shape', '65')
('3.2', 'Relaxation, Reconstruction, and Defects', '71')
('3.3.1', 'Surface Lattices and Superstructures', '76')
('3.3.2', '2D Reciprocal Lattice', '79')
('3.4', 'Structural Models of Solid–Solid Interfaces', '80')
('3.5', 'Nucleation and Growth of Thin Films', '85')
('3.5.1', 'Modes of Film Growth', '85')
('3.5.2', '“Capillary Model” of Nucleation', '89')
('Panel V

In [87]:
lines = index_text.splitlines()
for i, line in enumerate(lines):
    m = chapter_regex.match(line)
    print(f"{i:3d} {'✓' if m else '✗'} {repr(line)}")

  0 ✗ 'Contents'
  1 ✓ '1 Surface and Interface Physics: Its Definition and Importance . . . 1'
  2 ✓ 'Panel I: Ultrahigh Vacuum (UHV) Technology . . . . . . . . . . 6'
  3 ✓ 'Panel II: Basics of Particle Optics and Spectroscopy . . . . . . . . 17'
  4 ✗ 'Problems . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 28'
  5 ✗ '2 Preparation of Well-Defined Surfaces, Interfaces and Thin Films . 29'
  6 ✓ '2.1 Why Is Ultrahigh Vacuum Used? . . . . . . . . . . . . . . . . . 29'
  7 ✓ '2.2 Cleavage in UHV . . . . . . . . . . . . . . . . . . . . . . . . . 31'
  8 ✓ '2.3 Ion Bombardment and Annealing . . . . . . . . . . . . . . . . . 34'
  9 ✓ '2.4 Evaporation and Molecular Beam Epitaxy (MBE) . . . . . . . . 35'
 10 ✓ '2.5 Epitaxy by Means of Chemical Reactions . . . . . . . . . . . . . 43'
 11 ✓ 'Panel III: Auger Electron Spectroscopy (AES) . . . . . . . . . . . 49'
 12 ✓ 'Panel IV: Secondary Ion Mass Spectroscopy (SIMS) . . . . . . . . 55'
 13 ✗ 'Problems . . . . . . . . . . .

In [44]:
section_titles = [i[1] for i in matches]

page_delta = last_index_page
section_start_pages = [(int(i[2])) + page_delta for i in matches]
section_end_pages = section_start_pages[1:] + [None]

sections = list(zip(section_titles, section_start_pages, section_end_pages))

#for section in sections:
#    print(section)

In [45]:
section_selector = widgets.Dropdown(
    options=[(f"{title} (starts at page {start})", (start, end)) for title, start, end in sections],
    description="Section:",
)

output_widget = widgets.Output()

def on_section_selected(change):
    if change["type"] == "change" and change["name"] == "value":
        start_page, end_page = change["new"]
        with output_widget:
            print(f"Selected section starts at page {start_page} and ends at page {end_page}")

section_selector.observe(on_section_selected)

display(section_selector)
display(output_widget)

Dropdown(description='Section:', options=(('Surface and Interface Physics: Its Definition and Importance (star…

Output()

In [50]:
prompt = """

Analizza il seguente estratto di un libro di fisica delle superfici
e restituisci un oggetto JSON con questa struttura esatta.
Restituisci SOLO il JSON, nessun testo aggiuntivo, nessun backtick.

{
  "title": "titolo dell'estratto come appare nel testo",
  "summary": "riassunto sintetico ma completo dell'estratto, in italiano, scritto in markdown con equazioni LaTeX quando necessario",
  "key_concepts": ["lista dei concetti chiave trattati nell'estratto. Riporta solo i concetti più importanti e distintivi, non più di 10"],
  "key_equations": ["lista delle equazioni principali nell'estratto, in formato LaTeX"],
}


Estratto:

"""

pages = section_selector.value

texts = [i.extract_text(x_tolerance=2) for i in pdf.pages[pages[0]:pages[1]]]
text = "\n".join(texts)

prompt = prompt + "\n" + text

print(prompt)





Analizza il seguente estratto di un libro di fisica delle superfici
e restituisci un oggetto JSON con questa struttura esatta.
Restituisci SOLO il JSON, nessun testo aggiuntivo, nessun backtick.

{
  "title": "titolo dell'estratto come appare nel testo",
  "summary": "riassunto sintetico ma completo dell'estratto, in italiano, scritto in markdown con equazioni LaTeX quando necessario",
  "key_concepts": ["lista dei concetti chiave trattati nell'estratto. Riporta solo i concetti più importanti e distintivi, non più di 10"],
  "key_equations": ["lista delle equazioni principali nell'estratto, in formato LaTeX"],
}


Estratto:


5.3 Rayleigh Waves 225
Fig. 5.6 Relation between the 2D surface Brillouin zones of the (100), (111) and (110) surfaces
of a bcc lattice and the bulk Brillouin zone
u = r (cid:3) −r of these volume elements dv and the strain tensor
(cid:2) (cid:3)
1 ∂u ∂u
(cid:27) = j + i . (5.21)
ij
2 ∂x ∂x
i j
In the elastic regime (cid:27) is related to the stress field σ = ∂F

In [ ]:

client = None

with open(".env/key", "r") as f:
    key = f.read().strip()
    client = anthropic.Anthropic(api_key=key)

response = client.messages.create(
    model="claude-sonnet-4-6",
    #cache_control={"type": "ephemeral"},
    max_tokens=2048,
    messages=[
        {"role": "user", "content": prompt}
    ]
)

In [56]:
response_text = (response.content[0].text).replace("-", "").replace("json", "").replace("JSON", "").replace("```", "")
print(response_text)


{
  "title": "Rayleigh Waves",
  "summary": "Questo estratto tratta le onde di Rayleigh, onde elastiche di superficie in un mezzo continuo semiinfinito. Si parte dalla decomposizione del campo di spostamento in componenti longitudinali e trasversali, espresse tramite potenziali scalari $\\phi$ e vettoriali $\\psi$. Le equazioni d'onda per questi potenziali sono risolte con ansatz di onde superficiali che si propagano parallelo alla superficie con ampiezza decrescente verso il bulk. Le condizioni al contorno di assenza di stress elastico alla superficie determinano la relazione di dispersione. Si ricava che la velocità di fase dell'onda di Rayleigh $c_{RW} \\approx (11/24)c_t$ è inferiore alla velocità del suono trasversale. L'estratto evidenzia il carattere misto longitudinaletrasversale di queste onde e la loro natura non dispersiva nel limite continuo. Segue una descrizione dell'applicazione pratica dei filtri ad alta frequenza basati su onde di Rayleigh su cristalli piezoelettrici,

In [57]:

parsed_response = json.loads(response_text)
print(json.dumps(parsed_response, indent=2, ensure_ascii=False))


{
  "title": "Rayleigh Waves",
  "summary": "Questo estratto tratta le onde di Rayleigh, onde elastiche di superficie in un mezzo continuo semiinfinito. Si parte dalla decomposizione del campo di spostamento in componenti longitudinali e trasversali, espresse tramite potenziali scalari $\\phi$ e vettoriali $\\psi$. Le equazioni d'onda per questi potenziali sono risolte con ansatz di onde superficiali che si propagano parallelo alla superficie con ampiezza decrescente verso il bulk. Le condizioni al contorno di assenza di stress elastico alla superficie determinano la relazione di dispersione. Si ricava che la velocità di fase dell'onda di Rayleigh $c_{RW} \\approx (11/24)c_t$ è inferiore alla velocità del suono trasversale. L'estratto evidenzia il carattere misto longitudinaletrasversale di queste onde e la loro natura non dispersiva nel limite continuo. Segue una descrizione dell'applicazione pratica dei filtri ad alta frequenza basati su onde di Rayleigh su cristalli piezoelettrici, 